# Day 080 — Exercise 5: ReactAgent

**What you'll build:** the `ReactAgent` class — binds tools + `llm_fn`, runs tasks, and keeps a history (the same class shape as Day 79's `SimpleAgent`).

**Why it matters:** the reasoning loop is the engine; the class is the convenient handle. Bind the model and tools once, call `.run()` many times, and inspect `result['trace']` to see the agent's reasoning.

In [ ]:
import json

def _make_mock_llm(script):
    """Return an llm_fn(messages) that yields each scripted reply in turn.

    Repeats the last reply once the script is exhausted - handy for testing a
    runaway loop (a model that never emits a Final Answer).
    """
    state = {'i': 0}
    def _fn(messages):
        i = state['i']
        state['i'] = min(i + 1, len(script) - 1)
        return script[i]
    return _fn
import ast
import json
import operator

# ── tools reused from Day 79: a safe calculator + a fact-lookup tool ──────────
_OPS = {
    ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
    ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
    ast.USub: operator.neg, ast.UAdd: operator.pos,
}


def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError("unsupported expression")


def safe_calculate(expression):
    """Evaluate arithmetic without eval() (see Day 79)."""
    return _eval_node(ast.parse(expression, mode="eval").body)


_FACTS = {
    "speed of light": "299792458 m/s",
    "pi": "3.14159",
    "earth radius": "6371 km",
    "days in a year": "365",
}


def _lookup(args):
    query = str(args.get("query", "")).lower().strip()
    for key, value in _FACTS.items():
        if query and (query in key or key in query):
            return value
    return "No result found for " + repr(args.get("query", ""))


DEFAULT_TOOLS = {
    "calculator": {
        "description": "Evaluate an arithmetic expression, e.g. 2 * (3 + 4).",
        "parameters": {"expression": "string - the arithmetic to evaluate"},
        "fn": lambda args: str(safe_calculate(args["expression"])),
    },
    "lookup": {
        "description": "Look up a known fact: speed of light, pi, earth radius, "
                       "days in a year.",
        "parameters": {"query": "string - what to look up"},
        "fn": _lookup,
    },
}


def build_tool_descriptions(tools):
    """Render a tool registry as prompt text (Day 79)."""
    lines = []
    for name, spec in tools.items():
        params = ", ".join(spec.get("parameters", {}))
        lines.append("- " + name + "(" + params + "): " + spec["description"])
    return "\n".join(lines)


def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None

# ── parsing the ReAct format ──────────────────────────────────────────────────
def _line_value(text, prefix):
    """Text after the first line starting with prefix (case-insensitive), else ''."""
    for line in text.splitlines():
        if line.strip().lower().startswith(prefix.lower()):
            return line.strip()[len(prefix):].strip()
    return ""


def _after_marker(text, marker):
    """Everything after marker (case-insensitive), or None if absent."""
    idx = text.lower().find(marker.lower())
    if idx == -1:
        return None
    return text[idx + len(marker):].strip()


def parse_react_step(text):
    """Parse one ReAct step. NEVER raises.

    Returns either:
      {"type": "action", "thought": str, "tool": str, "input": dict}
      {"type": "final",  "thought": str, "answer": str}
    A reply with no recognisable Action falls back to a final answer holding
    the raw text - so a malformed step still ends the loop cleanly.
    """
    thought = _line_value(text, "Thought:")
    final = _after_marker(text, "Final Answer:")
    if final is not None:
        return {"type": "final", "thought": thought, "answer": final}
    action = _line_value(text, "Action:")
    if action:
        args = safe_parse_json(_line_value(text, "Input:")) or {}
        return {"type": "action", "thought": thought, "tool": action, "input": args}
    return {"type": "final", "thought": thought, "answer": text.strip()}

# ── formatting the trace (the scratchpad) ─────────────────────────────────────
def format_step(step):
    """Render an action step back into ReAct text for the scratchpad."""
    return ("Thought: " + step["thought"] + "\n"
            + "Action: " + step["tool"] + "\n"
            + "Input: " + json.dumps(step["input"]))


def format_observation(result):
    """Render a tool result as an Observation line."""
    return "Observation: " + str(result)


def build_react_prompt(task, tools, scratchpad):
    """Build the [system, user] messages for one ReAct step."""
    system = "\n".join([
        "You are a reasoning agent. Solve the task step by step using the "
        "ReAct format: reason, act, observe, repeat.",
        "",
        "Available tools:",
        build_tool_descriptions(tools),
        "",
        "On each turn reply in EXACTLY this format:",
        "Thought: <your reasoning about what to do next>",
        "Action: <one tool name from the list above>",
        'Input: {"<param>": "<value>"}',
        "",
        "You will then receive an Observation with the tool's result.",
        "When you can answer, reply instead with:",
        "Thought: <your final reasoning>",
        "Final Answer: <the answer>",
    ])
    user = "Task: " + str(task)
    if scratchpad:
        user = user + "\n\n" + scratchpad.rstrip()
    user = user + "\n\nThought:"
    return [{"role": "system", "content": system},
            {"role": "user", "content": user}]

# ── acting + calling the model ────────────────────────────────────────────────
def execute_action(step, tools):
    """Run the tool named in a ReAct action step. Returns a string, never raises."""
    name = step.get("tool")
    if name not in tools:
        return "Error: unknown tool " + repr(name) + ". Available: " + ", ".join(tools)
    try:
        return str(tools[name]["fn"](step.get("input", {})))
    except Exception as exc:
        return "Error running " + str(name) + ": " + str(exc)


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]

# ── the ReAct loop ────────────────────────────────────────────────────────────
def run_react_agent(task, tools=None, llm_fn=None, max_iterations=10):
    """Run the ReAct loop until a Final Answer or max_iterations.

    Each turn: build a prompt from the task + scratchpad, ask the model for a
    Thought/Action/Input, run the tool, append the step and its Observation to
    the scratchpad, repeat. Feeding the growing scratchpad back is what lets the
    model reason over its own earlier observations.

    Returns {"answer", "thought", "trace", "iterations", "stopped"}.
    """
    if tools is None:
        tools = DEFAULT_TOOLS
    scratchpad = ""
    trace = []
    for i in range(max_iterations):
        messages = build_react_prompt(task, tools, scratchpad)
        step = parse_react_step(call_llm(messages, llm_fn=llm_fn))
        if step["type"] == "final":
            trace.append(step)
            return {"answer": step["answer"], "thought": step["thought"],
                    "trace": trace, "iterations": i + 1, "stopped": False}
        result = execute_action(step, tools)
        step["observation"] = result
        trace.append(step)
        scratchpad = scratchpad + format_step(step) + "\n"
        scratchpad = scratchpad + format_observation(result) + "\n"
    return {"answer": "Stopped: reached max_iterations without a final answer.",
            "thought": "", "trace": trace,
            "iterations": max_iterations, "stopped": True}


## Task

`ReactAgent(tools=None, llm_fn=None, max_iterations=10)`

1. `__init__` — `self.tools = dict(DEFAULT_TOOLS if tools is None else tools)` (**copy**), plus `_llm_fn`, `max_iterations`, `_history = []`.
2. `add_tool(name, description, fn, parameters=None)` — add to `self.tools`; return `self`.
3. `run(task)` — delegate to `run_react_agent` with the bound tools/llm_fn; append `{'task','result'}` to `_history`; return the result.
4. `history()` — return `list(self._history)`.
5. `clear_history()` — `self._history.clear()`.

## Your Implementation

In [ ]:
class ReactAgent:
    """A reasoning agent using the ReAct loop."""

    def __init__(self, tools=None, llm_fn=None, max_iterations=10):
        raise NotImplementedError

    def add_tool(self, name, description, fn, parameters=None):
        raise NotImplementedError

    def run(self, task):
        raise NotImplementedError

    def history(self):
        raise NotImplementedError

    def clear_history(self):
        raise NotImplementedError


In [ ]:

# ── the ReAct agent as a class ────────────────────────────────────────────────
class ReactAgent:
    """A reasoning agent using the ReAct loop.

    Binds a tool registry and an optional llm_fn, runs tasks through
    run_react_agent, and keeps a history of runs.

    Example::

        agent = ReactAgent(llm_fn=my_llm_fn)
        result = agent.run("What is 12 * 12?")
        print(result["answer"])
        for step in result["trace"]:
            print(step)
    """

    def __init__(self, tools=None, llm_fn=None, max_iterations=10):
        # copy so add_tool never mutates the shared DEFAULT_TOOLS global (Day 79)
        self.tools = dict(DEFAULT_TOOLS if tools is None else tools)
        self._llm_fn = llm_fn
        self.max_iterations = max_iterations
        self._history = []

    def add_tool(self, name, description, fn, parameters=None):
        """Register a new tool; returns self."""
        self.tools[name] = {"description": description,
                            "parameters": parameters or {}, "fn": fn}
        return self

    def run(self, task):
        """Run one task through the ReAct loop. Returns the result dict."""
        result = run_react_agent(task, tools=self.tools, llm_fn=self._llm_fn,
                                 max_iterations=self.max_iterations)
        self._history.append({"task": task, "result": result})
        return result

    def history(self):
        """Return a copy of the run history."""
        return list(self._history)

    def clear_history(self):
        """Clear the run history in place."""
        self._history.clear()


## Automated checks

In [ ]:

score, total = 0, 6
try:
    script = ['Thought: add.\nAction: calculator\nInput: {"expression": "2+2"}',
              'Thought: done.\nFinal Answer: The answer is 4.']
    agent = ReactAgent(llm_fn=_make_mock_llm(script))
    out = agent.run('what is 2+2?')
    assert out['answer'] == 'The answer is 4.'
    score += 1; print("✅ ReactAgent.run returns the final answer")

    assert 'calculator' in agent.tools and 'lookup' in agent.tools
    score += 1; print("✅ ReactAgent uses DEFAULT_TOOLS by default")

    agent.add_tool('shout', 'Uppercase text.',
                   lambda args: str(args['text']).upper(), {'text': 'string'})
    assert 'shout' in agent.tools and 'shout' not in DEFAULT_TOOLS
    score += 1; print("✅ add_tool registers a tool without mutating DEFAULT_TOOLS")

    assert len(agent.history()) == 1
    score += 1; print("✅ history records each run")

    h = agent.history(); h.clear()
    assert len(agent.history()) == 1
    score += 1; print("✅ history() returns a copy, not the live list")

    agent.clear_history()
    assert len(agent.history()) == 0
    score += 1; print("✅ clear_history empties the log")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── the ReAct agent as a class ────────────────────────────────────────────────
class ReactAgent:
    """A reasoning agent using the ReAct loop.

    Binds a tool registry and an optional llm_fn, runs tasks through
    run_react_agent, and keeps a history of runs.

    Example::

        agent = ReactAgent(llm_fn=my_llm_fn)
        result = agent.run("What is 12 * 12?")
        print(result["answer"])
        for step in result["trace"]:
            print(step)
    """

    def __init__(self, tools=None, llm_fn=None, max_iterations=10):
        # copy so add_tool never mutates the shared DEFAULT_TOOLS global (Day 79)
        self.tools = dict(DEFAULT_TOOLS if tools is None else tools)
        self._llm_fn = llm_fn
        self.max_iterations = max_iterations
        self._history = []

    def add_tool(self, name, description, fn, parameters=None):
        """Register a new tool; returns self."""
        self.tools[name] = {"description": description,
                            "parameters": parameters or {}, "fn": fn}
        return self

    def run(self, task):
        """Run one task through the ReAct loop. Returns the result dict."""
        result = run_react_agent(task, tools=self.tools, llm_fn=self._llm_fn,
                                 max_iterations=self.max_iterations)
        self._history.append({"task": task, "result": result})
        return result

    def history(self):
        """Return a copy of the run history."""
        return list(self._history)

    def clear_history(self):
        """Clear the run history in place."""
        self._history.clear()
```

**Same shape as SimpleAgent?** Yes — bind at construction, delegate in methods, copy the registry, return a history copy. Every agent in this section wears the same class skeleton; only the loop inside changes.

</details>